# Predicting Loans (Machine Learning)

Our goal up to this point was to generate features from the data that will feed into a machine learning algorithm. The algorithm will make predictions about whether or not a loan will be paid off on time, which is contained in the ```loan_status``` column of the clean dataset.

As we prepared the data, we removed columns that had data leakage issues (information from the future that an investor would not have had), contained redundant information, or required additional processing to turn into useful features. We cleaned features that had formatting issues, and converted categorical columns to dummy variables.

**Class Imbalance** is a potential issue that came up during feature engineering that we need to be mindful of. There are about 6 times as many loans that were paid off on time (positive case, label 1) than those that weren't (negative case, label 0). Imbalances can cause issues with many machine learning algorithms, where they appear to have high accuracy, but actually aren't learning from the training data. Because of its potential to cause issues, we need to keep the class imbalance in mind as we build machine learning models.

In [145]:
## Read in libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

## Prevent annoying warning messages
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [146]:
df = pd.read_csv('loans_final.csv')
df.shape

(37675, 38)

In [147]:
df.head()

,loan_amnt,int_rate,installment,emp_length,annual_inc,loan_status,dti,delinq_2yrs,inq_last_6mths,open_acc,...,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,term_ 36 months,term_ 60 months
0,5000.0,10.65,162.87,10,24000.0,1,27.65,0.0,1.0,3.0,...,0,0,0,0,0,0,0,0,1,0
1,2500.0,15.27,59.83,0,30000.0,0,1.00,0.0,5.0,3.0,...,0,0,0,0,0,0,0,0,0,1
2,2400.0,15.96,84.33,10,12252.0,1,8.72,0.0,2.0,2.0,...,0,0,0,0,0,1,0,0,1,0
3,10000.0,13.49,339.31,10,49200.0,1,20.00,0.0,1.0,10.0,...,0,0,0,1,0,0,0,0,1,0
4,5000.0,7.90,156.46,3,36000.0,1,11.20,0.0,3.0,9.0,...,0,0,0,0,0,0,0,1,1,0


## Error Metric Selection

Our goal as an investor is to make money. We want to fund enough loans that are paid off on time to offset our losses from loans that aren't paid off. An error metric will help us determine if our algorithm will make us money or lose us money.

Primarily we're concerned with false positives and false negatives. With a false positive, we predict that a loan will be paid off in time when in reality its defaulted on. This costs us money since we are funding loans that lose us money. With false negatives, we are denying loans that _would_ be paid off and we do not make potential money.

Becuase we are taking the approach of a conservative investor, our priority will be minimizing risk and avoiding as much false positives as possible.

In [148]:
## How bad is class imbalance?
df.loan_status.value_counts(normalize=True)

1    0.856961
0    0.143039
Name: loan_status, dtype: float64

**Class imbalance** will cause issues when solely using accuracy as a metric. This is because a classifier can predict 1 for every row, and still have high accuracy. To illustrate, say we predicted 1 for every row, our model would be right 85.7% of the time. 

```If we loaned say $1000 on average at 10% interest:
    -6 true positives implies a gain of $600
    -1 false negative implies a loss of $1000 
 We lose $400 at the end if the day. ```

In this case we should not rely on accuracy alone, instead we ought to use metrics that tell us the number of false positives and false negatives.

Specifically we should optimize for:
- high recall (true positive rate) ```fpr = fp / (fp + tn)```
- low fall-out (false positive rate) ```tpr = tp / (tp + fn)```

In English:
- True Positive Rate: "the percentage of loans that should be funded that I would fund".
- False Positive Rate: "the percentage of the loans that shouldn't be funded that I would fund".

## Logistic Regression

Logistic regression is a good model to start out with as it is: 
- quick to train 
- less prone to overfitting than more complex models
- easy to interpret

In [149]:
## Set up target and feature sets

y = df['loan_status']
X = df.drop('loan_status',axis=1)

## Model Before Class Imbalance Adjustment

In [150]:
from sklearn.linear_model import LogisticRegression

lgm = LogisticRegression()

lgm.fit(X,y)
predictions = lgm.predict(X)

In [151]:
from sklearn.model_selection import cross_val_predict

predictions = cross_val_predict(lgm, X, y, cv=3)
predictions = pd.Series(predictions)

# False positives.
fp_filter = (predictions == 1) & (df["loan_status"] == 0)
fp = len(predictions[fp_filter])

# True positives.
tp_filter = (predictions == 1) & (df["loan_status"] == 1)
tp = len(predictions[tp_filter])

# False negatives.
fn_filter = (predictions == 0) & (df["loan_status"] == 1)
fn = len(predictions[fn_filter])

# True negatives
tn_filter = (predictions == 0) & (df["loan_status"] == 0)
tn = len(predictions[tn_filter])
# Rates
tpr = tp  / (tp + fn)
fpr = fp  / (fp + tn)
print(tpr)
print(fpr)

0.9987920460880877
0.9962887363147152


In [152]:
from sklearn.metrics import classification_report as cr
from sklearn.metrics import confusion_matrix as cm

print(cr(y,predictions))
print(cm(y,predictions))

              precision    recall  f1-score   support

           0       0.34      0.00      0.01      5389
           1       0.86      1.00      0.92     32286

    accuracy                           0.86     37675
   macro avg       0.60      0.50      0.46     37675
weighted avg       0.78      0.86      0.79     37675

[[   20  5369]
 [   39 32247]]


## Model After Class Imbalance Adjustment

Though _we_ are not using accuracy as an error metric, the classifier is, and it isn't accounting for the imbalance in the classes. There are a few ways to get a classifier to correct for imbalanced classes. The two main ones are:

- Use oversampling and undersampling to ensure that the classifier gets input that has a balanced number of each class
- Tell the classifier to penalize misclassifications of the less prevalent class more than the other class

We will use a penalizer by tweaking the ```class_weight``` setting of LogisticRegression.

In [153]:
from sklearn.linear_model import LogisticRegression

lgm = LogisticRegression(class_weight='balanced')

lgm.fit(X,y)
predictions = lgm.predict(X)

In [154]:
from sklearn.model_selection import cross_val_predict

predictions = cross_val_predict(lgm, X, y, cv=3)
predictions = pd.Series(predictions)

# False positives.
fp_filter = (predictions == 1) & (df["loan_status"] == 0)
fp = len(predictions[fp_filter])

# True positives.
tp_filter = (predictions == 1) & (df["loan_status"] == 1)
tp = len(predictions[tp_filter])

# False negatives.
fn_filter = (predictions == 0) & (df["loan_status"] == 1)
fn = len(predictions[fn_filter])

# True negatives
tn_filter = (predictions == 0) & (df["loan_status"] == 0)
tn = len(predictions[tn_filter])
# Rates
tpr = tp  / (tp + fn)
fpr = fp  / (fp + tn)
print(tpr)
print(fpr)

0.6614941460695039
0.3854147337168306


In [155]:
from sklearn.metrics import classification_report as cr
from sklearn.metrics import confusion_matrix as cm

print(cr(y,predictions))
print(cm(y,predictions))

              precision    recall  f1-score   support

           0       0.23      0.61      0.34      5389
           1       0.91      0.66      0.77     32286

    accuracy                           0.65     37675
   macro avg       0.57      0.64      0.55     37675
weighted avg       0.81      0.65      0.71     37675

[[ 3312  2077]
 [10929 21357]]


Let's modify the ```class_weight``` to have the following penalty:

In [156]:
penalty = {
    0: 10,
    1: 1
}
lgm = LogisticRegression(class_weight=penalty)

This dictionary will impose a penalty of ```10``` for misclassifying a ```0```, and a penalty of ```1``` for misclassifying a ```1```.

In [157]:
lgm.fit(X,y)
predictions = lgm.predict(X)

In [158]:
from sklearn.model_selection import cross_val_predict

predictions = cross_val_predict(lgm, X, y, cv=3)
predictions = pd.Series(predictions)

# False positives.
fp_filter = (predictions == 1) & (df["loan_status"] == 0)
fp = len(predictions[fp_filter])

# True positives.
tp_filter = (predictions == 1) & (df["loan_status"] == 1)
tp = len(predictions[tp_filter])

# False negatives.
fn_filter = (predictions == 0) & (df["loan_status"] == 1)
fn = len(predictions[fn_filter])

# True negatives
tn_filter = (predictions == 0) & (df["loan_status"] == 0)
tn = len(predictions[tn_filter])
# Rates
tpr = tp  / (tp + fn)
fpr = fp  / (fp + tn)
print(tpr)
print(fpr)

0.24886947903115902
0.0937094080534422


This adjustment lowered the false positive rate at the expense of lowering our true positive rate. While we have fewer false positives, we're also missing opportunities to fund more loans and potentially make more money.  

This calls for a new model.

## Random Forest 

Random forests are able to work with nonlinear data, and learn complex conditionals. Logistic regressions are only able to work with linear data. Training a random forest algorithm may enable us to get more accuracy due to columns that correlate nonlinearly with ```loan_status```.

In [159]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

clf = RandomForestClassifier(class_weight='balanced', random_state=1)
clf.fit(X, y)

predictions = clf.predict(X)
print(roc_auc_score(y, predictions))

0.9770362110984165


In [160]:
predictions = cross_val_predict(lgm, X_test, y_test, cv=3)
predictions = pd.Series(predictions)

# False positives.
fp_filter = (predictions == 1) & (df["loan_status"] == 0)
fp = len(predictions[fp_filter])

# True positives.
tp_filter = (predictions == 1) & (df["loan_status"] == 1)
tp = len(predictions[tp_filter])

# False negatives.
fn_filter = (predictions == 0) & (df["loan_status"] == 1)
fn = len(predictions[fn_filter])

# True negatives
tn_filter = (predictions == 0) & (df["loan_status"] == 0)
tn = len(predictions[tn_filter])
# Rates
tpr = tp  / (tp + fn)
fpr = fp  / (fp + tn)
print(tpr)
print(fpr)

0.2166001596169194
0.24173228346456693


# Conclusion

Unfortunately, using a random forest classifier didn't improve our false positive rate. The model is likely weighting too heavily on the 1 class, and still mostly predicting 1s. We could fix this by applying a harsher penalty for misclassifications of 0s.

Ultimately, our best model had a false positive rate of nearly 9%, and a true positive rate of nearly 24%. For a conservative investor, this means that they make money as long as the interest rate is high enough to offset the losses from 9% of borrowers defaulting, and that the pool of 24% of borrowers is large enough to make enough interest money to offset the losses.

If we had randomly picked loans to fund, borrowers would have defaulted on 14.5% of them, and our model is better than that.